# S03 Solutions — Git, GitHub & Project Structure

**Python for Applied Physics** · Session 3 solutions

> Exercises 1–3, 5, and 8 are process-based (terminal workflow, README writing). The solutions here show the key commands and expected outputs. Exercises 4, 6, and 7 have Python code to verify.

---

## Solution 1 — Repository initialization

Terminal commands (in order):

```bash
mkdir applied-physics-python && cd applied-physics-python
git init
# (run .gitignore cell from lecture)
git add .gitignore
git commit -m "Add .gitignore for scientific Python project"
mkdir -p S01_ecosystem/notebooks S02_control_flow/notebooks S03_git_github/notebooks shared
# (copy notebooks, run shared/optics.py and requirements.txt cells)
git add .
git commit -m "Add S01-S03 notebooks, shared optics module, requirements"
```

Expected `git log --oneline` output:
```
a2b3c4d Add S01-S03 notebooks, shared optics module, requirements
e5f6g7h Add .gitignore for scientific Python project
```

## Solution 2 — Commit message observations

Running against a recent NumPy clone, typical patterns include:

1. **Imperative mood**: "Fix", "Add", "Remove", "Refactor", "Update" — not past tense
2. **Module prefix**: `[linalg]`, `DOC:`, `BUG:`, `MAINT:` — type and scope at a glance
3. **Issue reference**: `(#12345)` — links commit to the tracker
4. **Specificity**: "Fix incorrect broadcasting in np.add for 0-d arrays" not just "fix bug"
5. **Short first line**: almost always under 72 characters; details go in the body

## Solution 3 — Push to GitHub

```bash
ssh-keygen -t ed25519 -C "you@example.com"   # if not done already
# Add ~/.ssh/id_ed25519.pub to GitHub → Settings → SSH keys
ssh -T git@github.com   # verify

git remote add origin git@github.com:USERNAME/applied-physics-python.git
git push -u origin main
```

Expected `git remote -v` output:
```
origin  git@github.com:USERNAME/applied-physics-python.git (fetch)
origin  git@github.com:USERNAME/applied-physics-python.git (push)
```

## Solution 4 — Feature branch: `peak_irradiance`

In [ ]:
# The function to add to shared/optics.py

import math

def peak_irradiance(E_J, tau_s, w0_m):
    """
    Peak irradiance at the beam waist of a Gaussian pulse.

    Uses the relation for a Gaussian spatial and temporal profile:
    I_peak = 2 * E / (π · w0² · τ)

    Parameters
    ----------
    E_J : float
        Pulse energy [J].
    tau_s : float
        Pulse duration FWHM [s].
    w0_m : float
        Beam waist radius (1/e² intensity) [m].

    Returns
    -------
    float
        Peak irradiance [W/m²].

    Notes
    -----
    The factor of 2 arises from the 2D Gaussian spatial integration.
    Assumes transform-limited pulse (no chirp).
    """
    return 2 * E_J / (math.pi * w0_m**2 * tau_s)


# Verify
I = peak_irradiance(1e-6, 300e-15, 50e-6)
print(f"I_peak = {I:.3e} W/m²  (expected ~8.49e11 W/m²)")
print(f"I_peak = {I/1e4:.3e} W/cm²")

**Terminal commands for the branch workflow:**
```bash
git switch -c feature/peak-irradiance
# edit shared/optics.py to add peak_irradiance
git add shared/optics.py
git commit -m "Add peak_irradiance function to optics module"
git switch main
git merge feature/peak-irradiance
git branch -d feature/peak-irradiance
git push
```

## Solution 5 — Merge conflict

This exercise is process-based. The key insight:
- After `git merge` reports a conflict, `git status` shows which files need resolution
- VS Code's merge editor shows **Current** (your branch) and **Incoming** (the merging branch) side by side
- After accepting a resolution, `git add <file>` marks it resolved
- `git commit` finalizes the merge (message is auto-generated)
- After the merge, `git log --oneline --graph` shows the two diverging and re-joining lines

## Solution 6 — `shared/pulses.py` module

In [ ]:
# Contents of shared/pulses.py

pulses_module = '''"""shared/pulses.py

Functions for computing pulsed laser parameters:
peak power, average power, duty cycle, and transform-limited pulse duration.
"""

import math


def pulse_parameters(E_J, tau_s, rep_rate_Hz):
    """
    Compute key parameters of a pulsed laser from measurable quantities.

    Parameters
    ----------
    E_J : float
        Pulse energy [J].
    tau_s : float
        Pulse duration FWHM [s].
    rep_rate_Hz : float
        Repetition rate [Hz].

    Returns
    -------
    P_peak : float
        Peak power [W] (Gaussian pulse approximation, factor 0.94).
    P_avg : float
        Average power [W].
    duty_cycle : float
        Duty cycle (dimensionless).
    """
    P_peak     = 0.94 * E_J / tau_s
    P_avg      = E_J * rep_rate_Hz
    duty_cycle = tau_s * rep_rate_Hz
    return P_peak, P_avg, duty_cycle


def transform_limit(delta_lambda_nm, center_lambda_nm):
    """
    Transform-limited pulse duration for a Gaussian pulse.

    Uses the Gaussian time-bandwidth product: Δt · Δν = 0.4413.
    Frequency bandwidth: Δν ≈ (c / λ²) · Δλ.

    Parameters
    ----------
    delta_lambda_nm : float
        Spectral bandwidth FWHM [nm].
    center_lambda_nm : float
        Center wavelength [nm].

    Returns
    -------
    float
        Transform-limited pulse duration FWHM [fs].

    Notes
    -----
    For a sech² pulse shape, the time-bandwidth product is 0.3148.
    """
    TBP = 0.4413
    c   = 3e8
    dl  = delta_lambda_nm * 1e-9
    lc  = center_lambda_nm * 1e-9
    d_nu = (c / lc**2) * dl
    dt_s = TBP / d_nu
    return dt_s * 1e15
'''

import os
shared_dir = os.path.join("..", "..", "shared")
os.makedirs(shared_dir, exist_ok=True)
with open(os.path.join(shared_dir, "pulses.py"), "w") as f:
    f.write(pulses_module)

print("shared/pulses.py written.")

# Verify
import sys
sys.path.insert(0, shared_dir)
import importlib, pulses
importlib.reload(pulses)

P_peak, P_avg, dc = pulses.pulse_parameters(50e-9, 300e-15, 76e6)
print(f"Peak power    : {P_peak/1e3:.2f} kW")
print(f"Average power : {P_avg:.3f} W")
print(f"Transform limit (10 nm @ 800 nm): {pulses.transform_limit(10, 800):.1f} fs")

## Solution 7 — Repo archaeology

In [ ]:
import subprocess
import os
from collections import Counter

def repo_report(repo_path):
    """
    Generate a summary report of a Git repository.

    Parameters
    ----------
    repo_path : str
        Path to the repository root.

    Returns
    -------
    dict
        Repository statistics.
    """
    def git(args):
        result = subprocess.run(
            ["git"] + args, capture_output=True, text=True, cwd=repo_path
        )
        return result.stdout.strip()

    # Total number of commits
    count_str = git(["rev-list", "--count", "HEAD"])
    n_commits = int(count_str) if count_str.strip().isdigit() else 0

    # First and last commit dates
    all_dates = [d for d in git(["log", "--format=%ai", "--reverse"]).split("\n") if d]
    first_date = all_dates[0][:10] if all_dates else "N/A"
    last_date  = all_dates[-1][:10] if all_dates else "N/A"

    # Most prolific author
    authors = git(["log", "--format=%an"]).split("\n")
    author_counts = Counter(authors)
    top_author, top_count = author_counts.most_common(1)[0]

    # Tracked files
    all_files = [f for f in git(["ls-files"]).split("\n") if f]
    py_files   = [f for f in all_files if f.endswith(".py")]

    return {
        "total_commits":  n_commits,
        "first_commit":   first_date,
        "last_commit":    last_date,
        "top_author":     f"{top_author} ({top_count} commits)",
        "total_files":    len(all_files),
        "python_files":   len(py_files),
    }


# Test on the course repo
print("=== Course repository ===")
report = repo_report(os.path.join("..", ".."))
for key, val in report.items():
    print(f"  {key:<18}: {val}")

# Test on NumPy clone if available
numpy_path = "/tmp/numpy-temp"
if os.path.exists(numpy_path):
    print("\n=== NumPy repository ===")
    report_np = repo_report(numpy_path)
    for key, val in report_np.items():
        print(f"  {key:<18}: {val}")

## Solution 8 — README.md

A complete README example:

In [ ]:
import os

readme = """# Applied Physics Python Course

Python programming for physicists — a 10-session practical course for the Master in Applied Physics (lasers, ultrashort pulses, structured beams).

## Course context

This repository contains all lecture notebooks, exercises, and solutions for the Python programming component of the Master in Applied Physics. Topics range from Python fundamentals to NumPy, Matplotlib, SciPy, OOP, testing, and high-performance computing with Numba and GPU acceleration.

## Setup

```bash
# 1. Clone the repository
git clone git@github.com:USERNAME/applied-physics-python.git
cd applied-physics-python

# 2. Create and activate a virtual environment
python -m venv .venv
source .venv/bin/activate        # Linux/macOS
.venv\\Scripts\\activate          # Windows

# 3. Install dependencies
pip install -r requirements.txt

# 4. Open in VS Code
code .
```

## Repository structure

```
applied-physics-python/
├── shared/             Reusable Python modules (optics.py, pulses.py, ...)
├── S01_ecosystem/      Session 1: Python ecosystem, VS Code, environment setup
├── S02_control_flow/   Session 2: Control flow, functions, debugger
├── S03_git_github/     Session 3: Git, GitHub, project structure
├── S04_numpy/          Session 4: NumPy arrays and scientific computing
├── ...                 Sessions 5–10
└── requirements.txt    Python package dependencies
```

## Session overview

| Session | Topic | Status |
|---------|-------|--------|
| S01 | Ecosystem / VS Code / Setup | ✅ Complete |
| S02 | Control Flow / Functions / Debugger | ✅ Complete |
| S03 | Git / GitHub / Project Structure | ✅ Complete |
| S04 | NumPy | 🔄 In progress |
| S05–S10 | ... | 📅 Upcoming |

## How to run the notebooks

Open any `.ipynb` file in VS Code and press `Shift+Enter` to run cells, or click **Run All** at the top of the notebook.
"""

readme_path = os.path.join("..", "..", "README.md")
with open(readme_path, "w") as f:
    f.write(readme.strip())

print("README.md written. Commit with:")
print('  git add README.md')
print('  git commit -m "Add comprehensive project README"')
print('  git push')